# Create MCP Server and Client using FastMCP

## Objectives

- Use FastMCP to create MCP Servers with **tools**, **resources**, and **prompts**
- Connect to MCP Servers using **In-Memory**, **HTTP**, and **STDIO** transports
- Test MCP Servers with client connections and manual tool calling
- Create a multi-server client and ReAct agent to use all our MCP tools


## Initial Setup

In [1]:
import socket
import asyncio
from fastmcp import FastMCP, Client

In [ ]:
import os

# Create the actual directory path where files will be stored
def make_dir():
    if os.path.exists("path"):
        print("✓ Path directory already exists")
    else:
        print("✗ Path directory doesn't exist - creating it...")
        os.makedirs("path")
        print("✓ Path directory created")

In this JupyterLab environment, we have to be careful running HTTP servers. Running a server on a port that is in use or killing a server on a running port causes the Python kernel to crash and you will have to rerun all the code cells. To deal with this, let's make a helper function to check the port we want to run our MCP server on. 


Choose a port (default 8000) for the `PORT` variable and run the following cell:


In [3]:
PORT = 8000

def test_port(port=PORT):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(('127.0.0.1', port))
            return False
        except socket.error:
            return True

f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

In [4]:
def print_stream_info(read, write, _sid, verbose=False):
    """Print information about the read/write streams and session ID.
    """
    if verbose:
        print("READ (receives FROM server):")
        print(read)
        print()
        
        print("WRITE (sends TO server):")
        print(write)
        print()
        
        print("SESSION ID:")
        print(_sid())

## Creating a Calculator MCPServer


MCP servers behave like functions that AI agents can call to perform actions. Let's define and configure a basic MCP server.

- **FastMCP** creates the server instance with a name and instructions
- `@mcp.tool` decorator registers functions as callable tools
- `@mcp.prompt` decorator creates reusable prompt templates
- `@mcp.resource` decorator exposes data resources with URI patterns


This creates a complete MCP server named **CalculatorMCPServer** with mathematical tools (add, subtract), a document reading resource, and a code review prompt template.

### Create an MCP server object

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="CalculatorMCPServer",
    instructions="""
        This server provides Calculation tools.
        Call add() or subtract() for simple mathematical operations.
    """)
print('mcp object', mcp)

mcp object FastMCP('CalculatorMCPServer')


### Tools

**Tools** (Active Capabilities)
- Functions that AI agents can call to perform actions
- Similar to LangChain tools but networked and discoverable

In [ ]:
@mcp.tool
def add(a: int, b: int) -> int:
    """
    Add two integers together.

    Args:
        a (int): The first integer.
        b (int): The second integer.

    Returns:
        int: The sum of `a` and `b`.

    Example:
        >>> add(3, 5)
        8
    """
    return a + b


@mcp.tool
def subtract(a: int, b: int) -> int:
    """
    Subtract one integer from another.

    Args:
        a (int): The number to subtract from.
        b (int): The number to subtract.

    Returns:
        int: The result of `a - b`.

    Example:
        >>> subtract(10, 4)
        6
    """
    return a - b


### Resources

Resources expose read-only data to the AI via URI-based addresses. You define them with `@mcp.resource("scheme://path/{param}")`, where `{param}` is a **named path parameter** the AI fills in when requesting that resource — for example, `file:///endpoint/{name}`, `file:///docs/{name}` creates an address where `{name}` gets replaced with whatever document the AI wants to read.\
Think of it as a GET endpoint — the AI calls the URI, your function runs, and the data is returned.

The URI is purely for resource identification, name the URI path something that fits the type, style, and content of resource you are exposing.

Let's create two resources, one that returns a prewritten message template and another that returns the contents of an actual file.

In [7]:
@mcp.resource("file:///endpoint/{name}")
def return_template_document(name: str) -> str:
    """Read a document by name"""
    return f"Document contents of {name}"

In [8]:
make_dir() # Create the actual directory path where files will be stored

✗ Path directory doesn't exist - creating it...
✓ Path directory created


In [ ]:
## Download files and add these to path directory

# %%capture
# !wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/aNE__JjH4DLNEibuNpfDlg/examples.txt
# !wget -P path/ https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/tfoeGPInNoajVS0DSohdVg/README.txt

**Note:** The URI endpoint is the MCP address for requesting resources, while the path is the actual storage location on your system. They are related but distinct.


In [10]:
@mcp.resource("file://endpoint2/{name}")
def read_document(name: str) -> str:
    """Read a document by name from the path directory"""
    try:
        # Read from the actual file system path
        with open(f"path/{name}", "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"Document '{name}' not found in path directory"
    except Exception as e:
        return f"Error reading document: {str(e)}"

### Prompts

Prompts are consistent, reusable templates that can be called for simple, repetitive tasks. They capture domain expertise in a structured way, so instead of reinventing instructions each time, the AI can rely on a proven pattern.

In [12]:
@mcp.prompt(title="Code Review")
def review_code(code: str) -> str:
    return f"Please review this code:\n\n{code}"

## Creating a Client - In-Memory Transport

Now let's create a client to test the server. We'll use **in-memory transport** — the simplest method where the server and client run in the same Python process, so no separate connection setup is needed. We use `async/await` because the client communicates with the server asynchronously.

**NOTE:** It's beneficial for testing but for production use HTTP Transport or STDIO Transport.

In [13]:
from fastmcp import  Client

client = Client(mcp)
print(f"client: {client}")

client: <fastmcp.client.client.Client object at 0x7f8a62b5b750>


### Calling Tools

Use `client.call_tool("tool_name", {args})` to invoke a tool on the server — even in-memory, it follows the MCP client-server pattern, so you're calling a server, not a local function.

Calls are wrapped in `async` because the server may take time to respond:
- `async with client` — opens and closes the connection automatically
- `await` — waits for the server response without blocking

In [14]:
async def call_add_tool(a: int, b: int):
    async with client:
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [15]:
response = await call_add_tool(4, 5)
response

CallToolResult(content=[TextContent(type='text', text='9', annotations=None, meta=None)], structured_content={'result': 9}, meta={'fastmcp': {'wrap_result': True}}, data=9, is_error=False)

#### Different ways to print response

In [18]:
# The actual answer/data
print("\nResult Data .data :")
print(response.data)  # 9

# Content (text format)
print("\nContent (as text):")
print(response.content[0].text)  # "9"

# Structured content (as dictionary)
print("\nStructured Content:")
print(response.structured_content)


Result Data .data :
9

Content (as text):
9

Structured Content:
{'result': 9}


`await client.list_tools()` returns all tools registered on the server — each with its name and description.

In [19]:
async with client:
    tools = await client.list_tools()
    print("Available tools:")
    for tool in tools:
        print(f"- {tool.name}: {tool.description}")
        print('-'*50)

Available tools:
- add: Add two integers together.

Args:
    a (int): The first integer.
    b (int): The second integer.

Returns:
    int: The sum of `a` and `b`.

Example:
    >>> add(3, 5)
    8
--------------------------------------------------
- subtract: Subtract one integer from another.

Args:
    a (int): The number to subtract from.
    b (int): The number to subtract.

Returns:
    int: The result of `a - b`.

Example:
    >>> subtract(10, 4)
    6
--------------------------------------------------


In [20]:
tool_obj = tools[0]
print(tool_obj)

name='add' title=None description='Add two integers together.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The sum of `a` and `b`.\n\nExample:\n    >>> add(3, 5)\n    8' inputSchema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'} outputSchema={'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True} icons=None annotations=None meta={'fastmcp': {'tags': []}} execution=None


In [21]:
input_schema = tool.inputSchema
print(input_schema)

{'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}


**Output Schema**: JSON structure defining what the tool returns — tells the client/LLM what to expect. MCP doesn't always expose this since some tools perform actions (like saving to database) where the LLM only needs simple confirmation, not detailed output data.

In [22]:
output_schema = tool.outputSchema
print(output_schema)

{'properties': {'result': {'type': 'integer'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}


### Reading Resources

Use `await client.read_resource(uri)` to fetch a resource — pass the full URI with the `{name}` parameter filled in (e.g. `file:///endpoint/myfile`).

In [23]:
async def call_resource(name):
    async with client:
        result = await client.read_resource(f"file:///endpoint/{name}")
        return result

In [24]:
response = await call_resource("README.txt")
print(response[0].text)

Document contents of README.txt


In [25]:
async def call_resource2(name):
    async with client:
        result = await client.read_resource(f"file://endpoint2/{name}")
        return result

In [29]:
response = await call_resource2("README.txt")
print(response[0].text)

# Documents Folder

This folder contains documents accessible through the Calculator MCP Server.

## Available Documents:

- examples.txt - Examples of how to use the calculator
- README.txt - This file



In [35]:
response = await call_resource2("random.txt")
resource = response[0]

print(f"uri:      {resource.uri}")
print(f"mimeType: {resource.mimeType}")
print(f"meta:     {resource.meta}")
print(f"text:     {resource.text}")

uri:      file://endpoint2/random.txt
mimeType: text/plain
meta:     None
text:     Document 'random.txt' not found in path directory


### Calling Prompts

Use `await client.get_prompt("prompt_name", {args})` to fetch a prompt — pass the code content as the argument.

In [ ]:
async def call_prompt(code):
    async with client:
        result = await client.get_prompt("review_code", {"code": code})
        return result

In [37]:
response = await call_prompt("CODE TO BE REVIEWED")

message=response.messages[0]
print(f"Prompt Role:{message.role}") # The role the prompt is intended for (e.g., 'user', 'assistant')
print(f"Prompt Content:{message.content.text}") # The actual prompt text

Prompt Role:user
Prompt Content:Please review this code:

CODE TO BE REVIEWED


## Creating a Client - HTTP Transport

Unlike in-memory, HTTP transport runs the server as a separate web service and the client connects to it via URL — useful for remote servers, cloud deployments, or sharing one server across multiple clients.

Check that the port is available (`True`) before starting the server. If unavailable, change `PORT` and rerun. *(JupyterLab only)*

In [38]:
f"Port {PORT} is available: {not test_port()}"

'Port 8000 is available: True'

### Starting HTTP MCP Server

`mcp.run_http_async(port=PORT)` starts the server on the specified port. It runs continuously in the background and auto-creates a `/mcp` endpoint for JSON-RPC communication.

**Jupyter only:** We wrap it in `asyncio.create_task()` so the server runs concurrently without blocking other cells.

**In a `.py` file:** Use `asyncio.run()` instead — it blocks until the server is stopped (Ctrl+C):
```python
if __name__ == "__main__":
    asyncio.run(mcp.run_http_async(port=8000))
```

In [ ]:
#server_task = asyncio.create_task(server_task_)
asyncio.create_task(mcp.run_http_async(port=PORT)) # to kill server simply restart kernel
print(f"HTTP MCP Server started in background on port {PORT}")

HTTP MCP Server started in background on port 8000




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 3.2.0                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      CalculatorMCPServer, 3.2.0                  │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                          

[04/08/26 18:28:21] INFO     Starting MCP server 'CalculatorMCPServer' with transport 'http' on    ]8;id=697023;file:///home/tk-lpt-648/miniconda3/envs/mcp_env/lib/python3.11/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=196234;file:///home/tk-lpt-648/miniconda3/envs/mcp_env/lib/python3.11/site-packages/fastmcp/server/mixins/transport.py#299\299]8;;\
                             http://127.0.0.1:8000/mcp                                                             

INFO:     Started server process [35156]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:56852 - "GET / HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:56852 - "GET /favicon.ico HTTP/1.1" 404 Not Found


 **IMPORTANT NOTE**

If you want to **change** or **rerun** the server at any point, you **MUST** change the port as to not crash the kernel.


### Creating the HTTP Client

Create an MCP client that uses **HTTP transport** to communicate with the server

In [40]:
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport
transport_http = StreamableHttpTransport(url=f"http://127.0.0.1:{PORT}/mcp")

Create the client using `transport_http` object

In [ ]:
from fastmcp import Client

http_client = Client(transport_http)
print('http_client', http_client)

http_client <fastmcp.client.client.Client object at 0x7f8a6210d190>


Same pattern as in-memory — `async with client` establishes a connection to the server, calls the tool with the provided arguments, and returns the result.

In [42]:
async def test_client_http(client: Client, a: int, b: int)->int:
    async with client: 
        # client establishes a connection with MCP server
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

In [43]:
response = await test_client_http(http_client, 4, 5)
print(response.content[0].text)

INFO:     127.0.0.1:54920 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54928 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:54934 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54948 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54952 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:54962 - "DELETE /mcp HTTP/1.1" 200 OK
9


**Note:** Calling tools, reading resources, and getting prompts over HTTP follows the exact same pattern as in-memory — the only difference is the transport the client connects through.

### LangChain ReAct Agent with HTTP MCP Server

To use MCP tools in LangChain, we use `langchain-mcp-adapters`. It requires a `ClientSession` (acts as the MCP client) paired with `StreamableHttpTransport` (handles the HTTP connection).

`load_mcp_tools()` converts the MCP tools into LangChain-compatible tools, which can then be passed to a `create_react_agent`.

In [44]:
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from mcp import ClientSession
llm = "openai:gpt-5-nano"

The ```streamable_http_client``` function connects you to an MCP server over HTTP. When you call it, it opens a connection and gives you back three things that let you communicate with the server:

```read:``` Receives messages from the server

```write:``` Sends messages to the server

```_sid:``` Returns your unique connection ID


In [45]:
from mcp.client.streamable_http import streamable_http_client
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    print_stream_info(read, write, _sid, verbose=True)

READ (receives FROM server):
MemoryObjectReceiveStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

WRITE (sends TO server):
MemoryObjectSendStream(_state=_MemoryObjectStreamState(max_buffer_size=0, buffer=deque([]), open_send_channels=1, open_receive_channels=1, waiting_receivers=OrderedDict(), waiting_senders=OrderedDict()), _closed=False)

SESSION ID:
None


Connect via `ClientSession`, load tools with `load_mcp_tools()`, create the agent, then invoke it. Use `ainvoke` (not `invoke`) since everything runs inside async context managers.

In [46]:
async with streamable_http_client(f"http://127.0.0.1:{PORT}/mcp") as (read, write, _sid):
    async with ClientSession(read, write) as session:
        await session.initialize()
        
        # Convert MCP tools into LangChain-compatible tools
        tools = await load_mcp_tools(session)
        
        # Build the agent WHILE the session is still open
        agent = create_react_agent(
            model=llm,
            tools=tools,
        )
        agent_response = await agent.ainvoke({"messages": "Use the add tool to add 2 and 1 and let me know if you used a tool."})

INFO:     127.0.0.1:36702 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36718 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:36730 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36734 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36748 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57714 - "DELETE /mcp HTTP/1.1" 200 OK


In [53]:
agent_response['messages'][-1].content

'Yes. I used the add tool to compute 2 + 1, and the result is 3. If you want more calculations, I can do them too.'

## Creating a Client - STDIO Transport

Unlike HTTP, STDIO transport spawns the server as a **child process** and communicates through `stdin/stdout` pipes — no URL, no running web server needed.

Since the server must run as a standalone process, we save it as a separate `.py` file in the same directory. The only difference from what we have written before is `mcp.run()` at the end, which tells the server to listen on stdin/stdout.

See `stdio_server.py` in the same directory for the server code.

### Configuring STDIO Transport

Now let's create a StdioTransport to communicate over STDIO transport by defining the command and arguments to launch the server. This will start the server using our ```stdio_server.py``` file.

The ```StdioTransport``` function specifies the command and arguments to launch the server:

```command="python"``` tells the system to use the Python interpreter

```args=["stdio_server.py"]``` provides the script to execute


In [ ]:
from fastmcp.client.transports import StdioTransport, StreamableHttpTransport
transport_stdio = StdioTransport(
    command="python",
    args=["stdio_server.py"]
)

**NOTE**: Relative paths work well in Jupyter when the notebook and server are in the same directory
Absolute paths are recommended for production environments for reliability


This `command/args` pattern is standard across MCP clients (e.g. Cursor, Claude Desktop), which use an `mcp.json` to configure both local (STDIO) and remote (HTTP) servers:

```json
{
    "mcpServers": {
        "local_stdio_server_name": {
            "command": "npx/python/uv/uvx",
            "args": ["some absolute/relative path", "some install argument"]
        },
        "remote_http_server_name": {
            "url": "mcp_server_url"
        }
    }
}
```

### Creating a STDIO Client

Same as HTTP — pass the `StdioTransport` into `Client()`. The difference is HTTP connects to an already-running server via URL, while STDIO spawns the server process itself on connection.

In [57]:
from fastmcp import Client

stdio_transport_client = Client(transport_stdio)
print(stdio_transport_client)

Same pattern as in-memory and HTTP — `async with client` spawns the server, calls the tool, and returns the result.

In [ ]:
async def test_client(client: Client, a: int, b: int) -> int:
    async with client:
        tools = await client.list_tools()
        result = await client.call_tool("add", {"a": a, "b": b})
        return result

response = await test_client(stdio_transport_client, 2, 5)
response_2 = await test_client_http(stdio_transport_client, 2, 5)
print(response.content[0].text)
print(response_2.content[0].text)

7
7


### LangChain ReAct Agent with STDIO MCP Server

Same pattern as HTTP — `ClientSession` + `load_mcp_tools()` + `create_react_agent`. The only difference is replacing `streamable_http_client` with `stdio_client`.

In [ ]:
# In an async context manager, convert and print the tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
# Configure the STDIO server parameters
server_params = StdioServerParameters(
    command="python",
    args=["stdio_server.py"],
)

Same as HTTP — create the agent, invoke with `ainvoke`. Async context managers keep the child process and session alive while the agent executes.

In [ ]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        agent = create_react_agent(
            model=llm,
            tools=tools,
        )

        agent_response = await agent.ainvoke(
            {"messages": "Use the add tool to add 2 and 1 ."}
        )

In [ ]:
print(agent_response['messages'][-1].content)

Yes. I used the add tool to compute 2 + 1, and the result is 3. If you want more calculations, I can do them too.


## Connecting Multiple MCP Servers

`MultiServerMCPClient` lets you connect to multiple MCP servers at once and load all their tools together. The transport config for each server is the same as we've used before — STDIO or HTTP.

In [61]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

In [62]:
client = MultiServerMCPClient(
    {
        "stdio-client": {
            "command": "python",
            "args": ["stdio_server.py"],
            "transport": "stdio"
        },
        "http-client": {
            "url": f"http://127.0.0.1:{PORT}/mcp",
            "transport": "streamable_http"
        }
    }
)

In [63]:
tools = await client.get_tools()
[tool.name for tool in tools]

INFO:     127.0.0.1:44728 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44740 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:44748 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44762 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:44770 - "DELETE /mcp HTTP/1.1" 200 OK


['add', 'subtract', 'add', 'subtract']

You should see 2 **add** and 2 **subtract** tools — one from each server, since both servers expose the same tools but over different transports.

### LangChain ReAct Agent with Multiple MCP Servers

Unlike the single-server approach, `MultiServerMCPClient` handles all connections internally — no manual `ClientSession`, no nested async context managers. Just get all tools with one call and pass them to `create_react_agent`.

**Note:** Prompt the agent to explicitly use the tools, otherwise the LLM may answer from its own knowledge instead.

In [64]:
llm = "openai:gpt-5-nano"
agent = create_react_agent(
    model=llm,
    tools=tools
)
agent_response = await agent.ainvoke({"messages": "whats 8 + 7? use tools"})

INFO:     127.0.0.1:49754 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49760 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:49768 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49770 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49780 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:49792 - "DELETE /mcp HTTP/1.1" 200 OK


In [65]:
for i in agent_response['messages']:
    if isinstance(i, HumanMessage):
        message_type = "HUMAN"
    elif isinstance(i, AIMessage):
        message_type = "AI"
    elif isinstance(i, ToolMessage):
        message_type = "TOOL"
    else:
        message_type = "OTHER"

    if i.content == '':
        i.content = "tool call"
    
    print(f"[{message_type}] {i.content}")

[HUMAN] whats 8 + 7? use tools
[AI] tool call
[TOOL] 15
[AI] The result is 15.


## Conclusion

In this lab, we built and tested a complete MCP server with **tools**, **resources**, and **prompts**, and connected to it using three transport methods:

- **In-Memory** — for quick testing within the same process
- **HTTP** — for remote or multi-client deployments
- **STDIO** — for local servers spawned as child processes

We also integrated MCP tools with LangChain agents using both single-server (`ClientSession`) and multi-server (`MultiServerMCPClient`) approaches. We now have everything needed to build and extend our own tool-enabled AI workflows.